# 11.13 多個字怎麼依序讀出？


圖片上寫1010，模型讀成1000，四個位置對了三個，卻沒有整串正確。對查驗編號的使用者，這一次仍可能失敗。多個字的OCR不只要逐字認對，還要保留順序、避免遺漏。本節把單字符表現與整串表現分開，讓指標貼近實際用途。

前置是[11.12字形與文字標籤](https://birdhackor.github.io/tiny-perceptron-vlm/11.12.html)和[W.2清單與迴圈](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.2)。exact match指完整字串完全相同才算成功。字符錯誤率CER是把預測改成真值所需的最少插入、刪除、替換次數，除以真值長度。本例只有一次替換，因此不用複雜算法也能手算1/4=0.25。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
truth = "1010"
predicted = "1000"
assert len(truth) == len(predicted)  # 本實驗只比較等長且對齊的字串
matched = sum(a == b for a, b in zip(truth, predicted))
print("等長逐位正確率", matched / len(truth))
print("整串正確", truth == predicted)
print("本例一次替換的CER", 1 / len(truth))
print("獨立錯誤假設下五位全對", round(0.9**5, 4))

輸入兩串長度4的文字。`zip`按相同位置產生一對一對的字，`for a, b`把每對拆成真值a與預測b，再用 `a == b`比較。`sum`加總這些結果時，把True當1、False當0；只有第三位真值1、預測0不同，因此matched=3。輸出0.75、False、0.25。這裡特意限制等長且位置對齊；若預測少一個字，不能用zip悄悄跳過末尾後仍叫完整CER，應使用能處理插入與刪除的編輯距離。

最後一行約0.5905：假設每位正確機率0.9，而且各位錯誤互不影響，五位同時正確的機率是五個0.9相乘。它解釋為什麼逐字90%不等於五位整串90%，但獨立只是例子假設。真實錯誤可能一起發生，例如裁切丟掉兩個末尾字，應該直接計算測試字串的exact match，不能靠這個乘法代替實測。

訓練目標要寫完整有序字串，讓模型逐步輸出直到[結束標記EOS](https://birdhackor.github.io/tiny-perceptron-vlm/7.9.html)。這是通知生成工具停止回答的特殊編號，並非圖片上需要讀出的文字；評估整串正確率與CER時，比較解碼後的文字，不把這個停止標記當成多出的字。前處理若切掉一半字，模型收到的資訊不足，必須先處理可見性問題。評估可同時報告整串正確率、CER與長度分組；長串通常更難，平均值可能掩蓋這種差異。

CER的分母用真值長度，所以真值1010、預測10100多一個0，也是一刪除、1/4=0.25；不是除預測長度5變0.2。若一個位置讀錯是替換，漏字是插入才能補齊，多字是刪除才能消掉。遇到錯位時，最少編輯可能不是逐位不同比例，例如預測010少了最前1，逐位對比看似多處錯誤，實際只需在開頭插入1。理解這個區別以後，再實現通用編輯距離才有明確目的；本節的等長小實驗特意沒有假裝完成它。

練習只把predicted改成"1010"，先預測逐位正確率1、整串True、CER0，再執行；CER那一行也應按這次已無編輯改成0。然後在紙上把預測寫成"101"，說明缺最後一個0需要一次插入，CER仍1/4，而整串仍False。程式的assert會拒絕等長實驗之外的情況，這個提醒比假裝算出了通用錯誤率更清楚。
